# Project D — Task **(a)**: Four top-level topics

**Assignment:** assign articles to the four top-level topics — Corporate/Industrial (**CCAT**), Economics (**ECAT**), Government/Social (**GCAT**), Markets (**MCAT**).

Same setup as the main project: topic tree from `topics.csv`; **binary relevance** per edge `Root → child`; gold from **`news_topics.csv`** (subtree membership). This notebook trains **H1 only**, then compares **LinearSVC**, **GLMNet-style elastic-net logistic**, and **MaxEnt L2 logistic** on **held-out test** metrics.

For **task (b)** (leaf / full hierarchy), see **`hierarchical_1b.ipynb`**.

**Training data per edge (local vs global):** `train_and_summarize`, `binary_edge_dataset`, and related helpers default to **parent-subtree–restricted** rows (`restrict_to_parent_subtree=True`): each edge uses only articles whose gold labels intersect the **parent** topic subtree. At Root this is usually still the full training set. Pass `restrict_to_parent_subtree=False` to `train_and_summarize` or `binary_edge_dataset` to reproduce the older **global** behavior (every edge saw all training documents).

**Working directory:** `Homework 4` (contains `topics.csv`).

In [8]:
from pathlib import Path

from topic_hierarchy import binary_edge_specs, load_topic_tree, summary


def homework4_base() -> Path:
    """Resolve Homework 4 folder (contains topics.csv)."""
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found — set cwd to Homework 4 or edit BASE")


BASE = homework4_base()
BASE

PosixPath('/Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4')

## Topic tree summary

In [9]:
tree = load_topic_tree(str(BASE / "topics.csv"))
s = summary(tree)
for k, v in sorted(s.items()):
    print(f"  {k}: {v}")

  max_branching_factor: 23
  max_depth_from_root: 5
  n_binary_edge_classifiers: 103
  n_branching_nodes: 22
  n_leaves: 82
  n_local_classifiers: 22
  n_nodes: 117
  traversal_root: Root


## Binary edge inventory (sample)

Each row is one trainable **binary** model: membership in the child’s subtree. Total count is **`n_binary_edge_classifiers`** in the summary above.

In [10]:
edges = binary_edge_specs(tree)
print(f"Total binary edges: {len(edges)}\n")
for sp in edges[:10]:
    print(f"  {sp.parent!r} -> {sp.child!r}  (depth {sp.depth})")

Total binary edges: 103

  'Root' -> 'CCAT'  (depth 0)
  'Root' -> 'ECAT'  (depth 0)
  'Root' -> 'GCAT'  (depth 0)
  'Root' -> 'MCAT'  (depth 0)
  'CCAT' -> 'C1'  (depth 1)
  'CCAT' -> 'C2'  (depth 1)
  'CCAT' -> 'C3'  (depth 1)
  'CCAT' -> 'C4'  (depth 1)
  'ECAT' -> 'E1'  (depth 1)
  'ECAT' -> 'E2'  (depth 1)


## H1 (Root): four branch classifiers — train / test / per-edge metrics

**Scope:** **first-level** edges under `Root` (CCAT, ECAT, GCAT, MCAT).

**Gold:** from `news_topics.csv` — for edge `Root → child`, label **1** iff the article has any topic in **subtree(`child`)**.

**Procedure (no k-fold):** fit each binary on the **full training** pool via `router.fit_edge`, then report metrics with **`evaluate_binary_edges_from_parent`** on the **same** fitted models — **train** split and **held-out test** split.

**Classifier:** **`LinearSVC`** (linear SVM) via `binary_edge_factory(..., estimator=LinearSVC, clf_kw=dict(C=..., dual=False, max_iter=...))`.

**Router:** `MultilabelHierarchyRouter.predict_reached_nodes` walks every predicted-positive branch (unary nodes pass through).


In [11]:
from sklearn.svm import LinearSVC

from hierarchical_training_data import make_multilabel_binary_pool_data
from hierarchical_classifier import MultilabelHierarchyRouter, binary_edge_factory
from hierarchical_evaluation import evaluate_binary_edges_from_parent

mldata = make_multilabel_binary_pool_data(base_path=str(BASE))

H1_PARENT = "Root"
h1_children = list(tree.children[H1_PARENT])
print("H1 branches:", h1_children)
print("Articles: train", len(mldata.train_ids()), "test", len(mldata.test_ids()))

edge_factory = binary_edge_factory(
    tfidf_kw=dict(min_df=2, max_features=8000),
    estimator=LinearSVC,
    clf_kw=dict(C=1.0, dual=False, max_iter=8000),
)

router = MultilabelHierarchyRouter(tree, edge_factory)

for child in h1_children:
    Xtr, ytr = mldata.binary_edge_dataset(H1_PARENT, child, "train")
    if len(set(ytr)) < 2:
        print(f"Skip {child}: need 2 classes in training")
        continue
    router.fit_edge(H1_PARENT, child, Xtr, ytr, depth=0)

metrics_train = evaluate_binary_edges_from_parent(router, mldata, H1_PARENT, split="train")
metrics_test = evaluate_binary_edges_from_parent(router, mldata, H1_PARENT, split="test")

print("\n--- Per-edge metrics (router models; full train fit) ---")
print("\nTrain:")
for child in sorted(metrics_train.keys()):
    print(f"\n[{H1_PARENT} → {child}]")
    print(" ", {k: round(v, 4) for k, v in metrics_train[child].items()})

print("\nTest (held-out):")
for child in sorted(metrics_test.keys()):
    print(f"\n[{H1_PARENT} → {child}]")
    print(" ", {k: round(v, 4) for k, v in metrics_test[child].items()})

H1 branches: ['CCAT', 'ECAT', 'GCAT', 'MCAT']
Articles: train 14465 test 3617

--- Per-edge metrics (router models; full train fit) ---

Train:

[Root → CCAT]
  {'accuracy': 0.9697, 'precision': 0.9675, 'recall': 0.9653, 'f1': 0.9664, 'roc_auc': 0.9954, 'avg_precision': 0.9946}

[Root → ECAT]
  {'accuracy': 0.9707, 'precision': 0.9706, 'recall': 0.9339, 'f1': 0.9519, 'roc_auc': 0.996, 'avg_precision': 0.9911}

[Root → GCAT]
  {'accuracy': 0.9811, 'precision': 0.9825, 'recall': 0.9745, 'f1': 0.9785, 'roc_auc': 0.9983, 'avg_precision': 0.9979}

[Root → MCAT]
  {'accuracy': 0.9838, 'precision': 0.9712, 'recall': 0.9115, 'f1': 0.9404, 'roc_auc': 0.998, 'avg_precision': 0.9881}

Test (held-out):

[Root → CCAT]
  {'accuracy': 0.9007, 'precision': 0.8878, 'recall': 0.8873, 'f1': 0.8876, 'roc_auc': 0.9622, 'avg_precision': 0.9526}

[Root → ECAT]
  {'accuracy': 0.9179, 'precision': 0.8829, 'recall': 0.8382, 'f1': 0.86, 'roc_auc': 0.9624, 'avg_precision': 0.9331}

[Root → GCAT]
  {'accuracy': 0.

### Sanity checks (gold labels + per-edge predictions)

**Gold:** `binary_edge_dataset` labels should match recomputing `gold_positive_edge(article_id, child)` for each pool id in order.

**Per-edge:** each H1 binary is independent — compare `predict_binary` on the same document across children.

In [12]:
CHECK_CHILD = "CCAT"
X_tr, y_tr = mldata.binary_edge_dataset(H1_PARENT, CHECK_CHILD, "train")
train_ids = mldata.train_ids()
y_check = [1 if mldata.gold_positive_edge(aid, CHECK_CHILD) else 0 for aid in train_ids]
n_mismatch = sum(1 for a, b in zip(y_tr, y_check) if a != b)
print(
    f"Edge {H1_PARENT} → {CHECK_CHILD}: label mismatches vs recomputed gold (train pool): "
    f"{n_mismatch} / {len(y_tr)}"
)
assert n_mismatch == 0, "binary_edge_dataset must match gold_positive_edge per article"

arts = mldata.articles_df().set_index("id")
sample_tid = mldata.test_ids()[0]
text = arts.loc[sample_tid, "article"]
print(f"\nSample test id={sample_tid}: per-edge predict_binary (Root → child)")
for ch in h1_children:
    m = router.edge_model(H1_PARENT, ch)
    if m is None:
        continue
    print(f"  {ch}: {m.predict_binary(text)}")

Edge Root → CCAT: label mismatches vs recomputed gold (train pool): 0 / 14465

Sample test id=2296: per-edge predict_binary (Root → child)
  CCAT: 1
  ECAT: 1
  GCAT: 1
  MCAT: 0


### Multi-branch inference (sample test articles)

`predict_reached_nodes` vs gold label sets from `news_topics`.

In [13]:
arts = mldata.articles_df().set_index("id")
for tid in mldata.test_ids()[:2]:
    text = arts.loc[tid, "article"]
    pred_nodes = router.predict_reached_nodes(text)
    gold = mldata.article_labels.get(tid, set())
    print("id", tid)
    print("  gold labels (sample):", sorted(gold)[:12], "..." if len(gold) > 12 else "")
    print("  predicted reached nodes (sample):", sorted(pred_nodes)[:12], "..." if len(pred_nodes) > 12 else "")

id 2296
  gold labels (sample): ['C13', 'C31', 'C42', 'CCAT', 'E41', 'ECAT', 'GCAT', 'GJOB'] 
  predicted reached nodes (sample): ['CCAT', 'ECAT', 'GCAT'] 
id 2348
  gold labels (sample): ['C11', 'C31', 'C312', 'CCAT'] 
  predicted reached nodes (sample): ['CCAT'] 


## Task 1a — Model comparison at **H1 (Root)** only: LinearSVC vs GLMNet vs MaxEnt

**Autext-style linear baselines (sklearn):**

| Preset | Role |
|--------|------|
| **LinearSVC** | Linear SVM on TF-IDF |
| **GLMNet_elasticnet** | Elastic-net *logistic* regression (`penalty='elasticnet'`, `solver='saga'`) — common GLMNet analogue |
| **MaxEnt_L2** | L2 *logistic* regression — binary/multinomial max-ent in NLP is typically log-linear (logistic) |

**Metrics (held-out test pool), H1 only:** for **F1**, **precision**, and **recall**: **macro** (mean over the four edges), **pooled micro** (one score on all H1 edge×document pairs), and **positive-support–weighted** (weighted by each edge’s positive count on the split). Also **pooled micro-accuracy** on those pairs, a **pooled 2×2 confusion matrix**, and a **full per-edge** confusion matrix for each `Root → child` branch.

Leaf / lowest-level metrics are **omitted** here (`include_leaf=False` in `train_and_summarize`) — handle those in a later part of the assignment if required.

Each row retrains **H1 only** (fresh router). Deeper edges (e.g. `GCAT → …`) are not part of this table.

In [14]:
import time

from IPython.display import display

from hierarchical_summary_metrics import (
    comparison_table,
    linear_model_factories,
    train_and_summarize,
)

# Task 1a: H1 (Root) metrics only — no leaf evaluation.
# train_and_summarize fits on the train pool; summary_row(..., split="test") scores on test only.
factories = linear_model_factories(max_features=8000)
n_models = len(factories)
print(
    f"Task 1a: {n_models} models. Fit each on **train**; every number below is **test (held-out)** H1.\n"
    f"Order: {list(factories.keys())}\n",
    flush=True,
)

summary_rows = []
for i, (model_name, factory) in enumerate(factories.items(), start=1):
    print(f"[{i}/{n_models}] {model_name}: fit train → eval test split ...", flush=True)
    t0 = time.perf_counter()
    _, row = train_and_summarize(
        model_name, tree, mldata, factory, split="test", include_leaf=False
    )
    elapsed = time.perf_counter() - t0
    summary_rows.append(row)
    print(
        f"    ok ({elapsed:.1f}s) test: h1_macro_f1={row['h1_macro_f1']:.4f}, "
        f"h1_pooled_micro_f1={row['h1_pooled_micro_f1']:.4f}, "
        f"h1_macro_precision={row['h1_macro_precision']:.4f}, "
        f"h1_macro_recall={row['h1_macro_recall']:.4f}\n",
        flush=True,
    )

print("Done. Summary table (test metrics only):", flush=True)
summary_df = comparison_table(summary_rows)
_skip = {"h1_pooled_confusion_matrix", "h1_per_edge_confusion"}
display_cols = [c for c in summary_df.columns if c not in _skip]
out = summary_df[display_cols].copy()
num_cols = [c for c in out.columns if c != "model"]
out[num_cols] = out[num_cols].round(4)
display(out)

print(
    "\nConfusion matrices (sklearn: rows = true class 0/1, cols = predicted 0/1). "
    "Pooled = all H1 edge×document pairs stacked.\n"
)
for row in summary_rows:
    name = row["model"]
    print(f"--- {name} ---")
    print("Pooled H1:")
    print(row["h1_pooled_confusion_matrix"])
    print("Per edge Root → child:")
    for (parent, child), cm in sorted(row["h1_per_edge_confusion"].items()):
        print(f"  {parent} → {child}:")
        print(cm)
    print()

Task 1a: 3 models. Fit each on **train**; every number below is **test (held-out)** H1.
Order: ['LinearSVC', 'GLMNet_elasticnet', 'MaxEnt_L2']

[1/3] LinearSVC: fit train → eval test split ...
    ok (42.6s) test: h1_macro_f1=0.8671, h1_pooled_micro_f1=0.8802, h1_macro_precision=0.8878, h1_macro_recall=0.8483

[2/3] GLMNet_elasticnet: fit train → eval test split ...
    ok (70.0s) test: h1_macro_f1=0.8575, h1_pooled_micro_f1=0.8786, h1_macro_precision=0.9097, h1_macro_recall=0.8148

[3/3] MaxEnt_L2: fit train → eval test split ...
    ok (34.1s) test: h1_macro_f1=0.8608, h1_pooled_micro_f1=0.8814, h1_macro_precision=0.9153, h1_macro_recall=0.8164

Done. Summary table (test metrics only):


,model,h1_macro_f1,h1_macro_precision,h1_macro_recall,h1_pooled_micro_f1,h1_pooled_micro_precision,h1_pooled_micro_recall,h1_pooled_micro_accuracy,h1_pos_weighted_f1,h1_pos_weighted_precision,h1_pos_weighted_recall
0,LinearSVC,0.8671,0.8878,0.8483,0.8802,0.8938,0.8671,0.9215,0.8798,0.8935,0.8671
1,GLMNet_elasticnet,0.8575,0.9097,0.8148,0.8786,0.9127,0.8469,0.9221,0.8770,0.9124,0.8469
2,MaxEnt_L2,0.8608,0.9153,0.8164,0.8814,0.9173,0.8482,0.9240,0.8798,0.9172,0.8482



Confusion matrices (sklearn: rows = true class 0/1, cols = predicted 0/1). Pooled = all H1 edge×document pairs stacked.

--- LinearSVC ---
Pooled H1:
[[9158  496]
 [ 640 4174]]
Per edge Root → child:
  Root → CCAT:
[[1841  179]
 [ 180 1417]]
  Root → ECAT:
[[2408  121]
 [ 176  912]]
  Root → GCAT:
[[1879  133]
 [ 163 1442]]
  Root → MCAT:
[[3030   63]
 [ 121  403]]

--- GLMNet_elasticnet ---
Pooled H1:
[[9264  390]
 [ 737 4077]]
Per edge Root → child:
  Root → CCAT:
[[1861  159]
 [ 191 1406]]
  Root → ECAT:
[[2454   75]
 [ 224  864]]
  Root → GCAT:
[[1899  113]
 [ 155 1450]]
  Root → MCAT:
[[3050   43]
 [ 167  357]]

--- MaxEnt_L2 ---
Pooled H1:
[[9286  368]
 [ 731 4083]]
Per edge Root → child:
  Root → CCAT:
[[1866  154]
 [ 183 1414]]
  Root → ECAT:
[[2459   70]
 [ 224  864]]
  Root → GCAT:
[[1907  105]
 [ 159 1446]]
  Root → MCAT:
[[3054   39]
 [ 165  359]]

